In [1]:
import sys

sys.path.append("../src")
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

driver_features = pd.read_csv("../outputs/driver_features.csv")
vehicle_health_features = pd.read_csv("../outputs/vehicle_health_features.csv")

print("driver_features:", driver_features.shape)
print("vehicle_health_features:", vehicle_health_features.shape)
driver_features.head()

driver_features: (30, 17)
vehicle_health_features: (30, 22)


,Driver_ID,accel_x_std,accel_y_std,accel_x_max_abs,gyro_z_std,gyro_z_max_abs,speed_mean,total_trips,mean_harsh_events_per_min,max_harsh_events_per_min,std_harsh_events_per_min,total_harsh_events,norm_accel_x_std,norm_gyro_z_std,norm_mean_harsh_rate,norm_max_harsh_rate,risk_score
0,D01,0.166228,0.097211,0.800,10.315407,53.97,25.226464,15,0.164273,0.263158,0.075844,71,0.539426,0.978655,0.534826,0.320618,0.593381
1,D02,0.144525,0.094453,0.748,7.611068,52.70,22.142537,15,0.145051,0.294118,0.093747,56,0.409247,0.545934,0.458496,0.384242,0.449479
2,D03,0.220968,0.104943,0.790,10.303619,54.48,22.660788,15,0.281415,0.500000,0.099874,127,0.867771,0.976769,1.000000,0.807339,0.912970
3,D04,0.118352,0.092586,0.744,9.315891,54.60,24.535579,15,0.144922,0.285714,0.081578,65,0.252256,0.818722,0.457983,0.366972,0.473983
4,D05,0.076297,0.070408,0.638,5.989514,45.79,23.112448,15,0.048010,0.192308,0.055203,24,0.000000,0.286469,0.073141,0.175018,0.133657


In [2]:
# Cluster drivers into risk tiers using the same normalized features
# that built the baseline risk_score, so results are directly comparable
cluster_features = [
    "norm_accel_x_std",
    "norm_gyro_z_std",
    "norm_mean_harsh_rate",
    "norm_max_harsh_rate",
]
X = driver_features[cluster_features].values

# Try k=2 through k=5 and compare silhouette scores to justify
# the number of clusters rather than picking one arbitrarily
for k in range(2, 6):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    score = silhouette_score(X, labels)
    print(f"k={k}: silhouette={score:.3f}")

k=2: silhouette=0.483
k=3: silhouette=0.516
k=4: silhouette=0.439
k=5: silhouette=0.394


In [3]:
# k=3 selected based on highest silhouette score (0.516) — not
# chosen for interpretability convenience, though it happens to
# map naturally onto Low/Medium/High risk tiers
kmeans_final = KMeans(n_clusters=3, random_state=42, n_init=10)
driver_features["cluster"] = kmeans_final.fit_predict(X)

# Map cluster IDs to meaningful labels by their mean risk_score,
# so labels are semantically ordered rather than arbitrary cluster numbers
cluster_order = (
    driver_features.groupby("cluster")["risk_score"].mean().sort_values().index
)
tier_map = {
    cluster_order[0]: "Low Risk",
    cluster_order[1]: "Medium Risk",
    cluster_order[2]: "High Risk",
}
driver_features["risk_tier"] = driver_features["cluster"].map(tier_map)

driver_features[["Driver_ID", "risk_score", "risk_tier"]].sort_values(
    "risk_score", ascending=False
)

,Driver_ID,risk_score,risk_tier
13,D14,0.946764,High Risk
2,D03,0.912970,High Risk
18,D19,0.846709,High Risk
5,D06,0.800270,High Risk
11,D12,0.794875,High Risk
22,D23,0.788951,High Risk
23,D24,0.770839,High Risk
16,D17,0.634270,Medium Risk
19,D20,0.615848,Medium Risk
0,D01,0.593381,Medium Risk


In [4]:
# Validate that clustering broadly agrees with the baseline rule-based
# score — if High Risk tier drivers also have the highest risk_score,
# the unsupervised method is confirming the interpretable baseline
# rather than contradicting it
print(
    driver_features.groupby("risk_tier")["risk_score"].describe()[
        ["count", "mean", "min", "max"]
    ]
)

             count      mean       min       max
risk_tier                                       
High Risk      7.0  0.837340  0.770839  0.946764
Low Risk      10.0  0.187525  0.072818  0.328470
Medium Risk   13.0  0.497764  0.394537  0.634270


Driver risk clustering — K-means, k=3

Number of clusters chosen via silhouette score comparison (k=2: 0.483, k=3: 0.516, k=4: 0.439, k=5: 0.394) — k=3 selected as the highest-scoring option, not chosen for interpretability convenience, though it maps naturally to Low/Medium/High risk tiers.

Clusters were mapped to tier labels by their mean risk_score, then validated against the baseline score itself: the three tiers show zero overlap (High Risk min 0.771 > Medium Risk max 0.634; Medium Risk min 0.395 > Low Risk max 0.328). This means the unsupervised clustering — using only the same four normalized features as the baseline score, without ever seeing risk_score directly — independently discovered the same ordering the rule-based score produced.

So what: this is a strong validation result. The baseline score isn't an arbitrary weighting scheme; the underlying structure in the data supports it. The dashboard can confidently present risk_tier as the primary driver-facing label (simpler, more actionable for a fleet manager than a raw 0-1 score), backed by both an interpretable baseline and an independent unsupervised method agreeing with it.

Tier distribution: High Risk (7 drivers), Medium Risk (13 drivers), Low Risk (10 drivers).